#First Step: Build The Protocol Manifests

This is Step 0. It creates the official `counterfact_direction1_v1` split files from `azhx/counterfact` before any model evaluation happens.

It should produce:

```text
dev_tune_200
analysis_500
ablation_500
final_test_500
final_test_full
```

with context-aware target tokenization, target-length bins, relation histograms, validity reports, and split-overlap checks.

Run this before any bridge/base/baseline experiment.

In [1]:
!pip install -q \
  "transformers==4.46.3" \
  "datasets>=4.0.0" \
  "accelerate>=1.11.0" \
  "sentencepiece>=0.2.0" \
  "packaging" \
  "torch"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 106.1 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### tiny smoke build first:


In [6]:
%%bash
cd /content/drive/MyDrive/Masters/SB\ V2/SB

python llada_counterfact_protocol.py \
  --dataset_name azhx/counterfact \
  --tokenizer_model_id GSAI-ML/LLaDA-8B-Base \
  --output_dir runs/counterfact_direction1_v1/protocol_smoke \
  --seed 0 \
  --smoke 1 \
  --max_train_rows 50 \
  --max_test_rows 20 \
  --skip_final_test_full 1

[INFO] Wrote protocol manifests to runs/counterfact_direction1_v1/protocol_smoke


### Inspect

In [7]:
%%bash
cd /content/drive/MyDrive/Masters/SB\ V2/SB

python - <<'PY'
import json

base = "runs/counterfact_direction1_v1/protocol_smoke"

with open(f"{base}/protocol_manifest.json") as f:
    manifest = json.load(f)

print("Protocol:", manifest["protocol_version"])
print("Smoke:", manifest["smoke"])
print("Split sizes:", manifest["split_sizes"])

for split, info in manifest["artifacts"].items():
    print(split, info["summary"]["count"], info["summary"]["target_length_bin_histogram"])

with open(f"{base}/split_overlap_report.json") as f:
    overlap = json.load(f)

print("Has disallowed overlap:", overlap["has_disallowed_overlap"])
PY


Protocol: counterfact_direction1_v1
Smoke: True
Split sizes: {'dev_tune_200': 10, 'analysis_500': 10, 'ablation_500': 10, 'final_test_500': 10, 'final_test_full': 0}
dev_tune_200 10 {'1': 9, '2': 1}
analysis_500 10 {'1': 9, '3': 1}
ablation_500 10 {'1': 10}
final_test_500 10 {'1': 9, '2': 1}
Has disallowed overlap: False


### Code

In [8]:
%%bash
cd /content/drive/MyDrive/Masters/SB\ V2/SB

python llada_counterfact_protocol.py \
  --dataset_name azhx/counterfact \
  --tokenizer_model_id GSAI-ML/LLaDA-8B-Base \
  --output_dir runs/counterfact_direction1_v1/protocol \
  --seed 0

[INFO] Wrote protocol manifests to runs/counterfact_direction1_v1/protocol


### What To Expect

The smoke run should finish quickly and print something like:

```text
[INFO] Wrote protocol manifests to runs/counterfact_direction1_v1/protocol_smoke
```

The real run should produce files like:

```text
runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl
runs/counterfact_direction1_v1/protocol/analysis_500.jsonl
runs/counterfact_direction1_v1/protocol/ablation_500.jsonl
runs/counterfact_direction1_v1/protocol/final_test_500.jsonl
runs/counterfact_direction1_v1/protocol/final_test_full.jsonl
runs/counterfact_direction1_v1/protocol/protocol_manifest.json
runs/counterfact_direction1_v1/protocol/split_overlap_report.json
runs/counterfact_direction1_v1/protocol/validity_report.json
```

### Then inspect the manifest and overlap report:

In [9]:
%%bash
cd /content/drive/MyDrive/Masters/SB\ V2/SB

python - <<'PY'
import json

base = "runs/counterfact_direction1_v1/protocol"

with open(f"{base}/protocol_manifest.json") as f:
    manifest = json.load(f)

print("Protocol:", manifest["protocol_version"])
for split, info in manifest["artifacts"].items():
    print(split, info["summary"]["count"], info["summary"]["target_length_bin_histogram"])

with open(f"{base}/split_overlap_report.json") as f:
    overlap = json.load(f)

print("Has disallowed overlap:", overlap["has_disallowed_overlap"])
PY

Protocol: counterfact_direction1_v1
dev_tune_200 200 {'1': 128, '2': 69, '3': 3}
analysis_500 500 {'1': 370, '2': 119, '3': 11}
ablation_500 500 {'1': 429, '2': 58, '3': 13}
final_test_500 500 {'1': 456, '2': 31, '3': 13}
final_test_full 2191 {'1': 2140, '2': 31, '3': 20}
Has disallowed overlap: False


Expected:

```text
Protocol: counterfact_direction1_v1
dev_tune_200 200 ...
analysis_500 500 ...
ablation_500 500 ...
final_test_500 500 ...
final_test_full around 2190 ...
Has disallowed overlap: False
```

If `has_disallowed_overlap` is `True`, stop there and inspect `split_overlap_report.json`. If it is `False`, the next step is base/self-consistency plus candidate-set coverage on `dev_tune_200`.